# 04 FIFA ranking

**Owner:** Sarthak

- Load the FIFA Men's World Ranking, check date coverage, keep **rank** (never points: the formula changed in 2018).
- The file has year + semester only. Semester 1 = 30 June, semester 2 = 31 December (decision in `DECISIONS.md`).
  Nation rank at entry = rank on the latest ranking date on or before the edition date, so every edition
  (all launched Aug–Sept) uses that year's semester 1 ranking.
- Map FIFA `nationality_name` to the ranking. Ranking team names change over time ("Turkey" became "Türkiye"), so the join key
  is the ranking's FIFA trigram (`acronym`), used **only** as an internal key and never as an ISO code.
- ISO3 codes for the Datawrapper map come from `pycountry` (plus a hand-checked list in `src/nations.py`). England, Scotland, Wales
  and Northern Ireland map to GBR.
- Confederation comes from Transfermarkt `national_teams.csv` (falling back to `countries.csv`), since the ranking has none.

**Manual file:** `data/manual/nation_name_overrides.csv` (`fifa_nation, ranking_team, tm_country, confederation, reviewer, note`).

**Outputs:** `data/clean/nation_rank.parquet` (FIFA nation × edition), `data/clean/nation_iso.parquet` (one row per FIFA nation),
section `04` of `reports/data_quality.md`.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from rapidfuzz import fuzz, process
from IPython.display import display

from src import config, dq, nations

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

CHECKS, NOTES = [], []
def check(name, passed, detail=""):
    CHECKS.append({"Check": name, "Result": "PASS" if passed else "FAIL", "Detail": detail})
    print(("PASS" if passed else "FAIL"), "-", name, ("| " + detail if detail else ""))
    return passed

## 1. Ranking: inspect, coverage, rank only

In [2]:
raw = pd.read_csv(config.RANKING_FILE)
print(raw.shape); display(raw.dtypes.to_frame("dtype").T); display(raw.head())

rk = raw.rename(columns={"date": "year"})[["year", "semester", "rank", "team", "acronym"]].copy()
rk["ranking_date"] = [nations.ranking_semester_date(y, s) for y, s in zip(rk["year"], rk["semester"])]
rk["key"] = rk["acronym"].replace(nations.ACRONYM_ALIASES)
print("Points columns dropped:", [c for c in raw.columns if "points" in c])

need = {(y, s) for y in range(2014, 2024) for s in (1, 2)}
have = set(zip(rk["year"], rk["semester"]))
check("Ranking covers every half-year 2014–2023", need <= have, f"missing: {sorted(need - have)}" if need - have else
      f"{rk['ranking_date'].min():%Y-%m-%d} to {rk['ranking_date'].max():%Y-%m-%d}")
dups = rk[rk.duplicated(["key", "year", "semester"], keep=False)]
if len(dups):
    NOTES.append(f"{len(dups)} duplicated (team, period) rows in the raw ranking, all outside 2014–2023: "
                 + ", ".join(sorted({f"{t} {y} S{s}" for t, y, s in zip(dups['team'], dups['year'], dups['semester'])}))
                 + ". Identical copies; they are never used.")
    print("Duplicated (team, period) rows:"); display(dups)
window = rk["year"].between(2014, 2023)
check("No duplicate (key, period) rows in 2014–2023", not rk[window].duplicated(["key", "year", "semester"]).any(),
      f"{len(dups)} duplicate rows elsewhere (logged)" if len(dups) else "")
check("Ranks are positive integers", bool((rk["rank"] >= 1).all()))

(13130, 8)


,date,semester,rank,team,acronym,total.points,previous.points,diff.points
dtype,int64,int64,int64,str,str,float64,float64,float64


,date,semester,rank,team,acronym,total.points,previous.points,diff.points
0,2024,2,1,Argentina,ARG,1867.25,1883.50,-16.25
1,2024,2,2,France,FRA,1859.78,1859.85,-0.07
2,2024,2,3,Spain,ESP,1853.27,1844.33,8.94
3,2024,2,4,England,ENG,1813.81,1807.83,5.98
4,2024,2,5,Brazil,BRA,1775.85,1784.37,-8.52


Points columns dropped: ['total.points', 'previous.points', 'diff.points']
PASS - Ranking covers every half-year 2014–2023 | 1992-12-31 to 2024-12-31
Duplicated (team, period) rows:


,year,semester,rank,team,acronym,ranking_date,key
7302,2007,2,172,Montenegro,MNE,2007-12-31,MNE
7303,2007,2,172,Montenegro,MNE,2007-12-31,MNE
7538,2007,1,199,Montenegro,MNE,2007-06-30,MNE
7548,2007,1,199,Montenegro,MNE,2007-06-30,MNE


PASS - No duplicate (key, period) rows in 2014–2023 | 4 duplicate rows elsewhere (logged)
PASS - Ranks are positive integers 


True

In [3]:
# Sanity check on the semester reading: who was ranked top in each semester-1 vs semester-2 list?
top = (rk[rk["year"].between(2014, 2023)].sort_values("rank").groupby(["year", "semester"]).head(1)
         .pivot(index="year", columns="semester", values="team").rename(columns={1: "S1 (30 Jun) #1", 2: "S2 (31 Dec) #1"}))
display(top)

semester,S1 (30 Jun) #1,S2 (31 Dec) #1
year,,
2014,Germany,Germany
2015,Argentina,Belgium
2016,Argentina,Argentina
2017,Germany,Germany
2018,Germany,Belgium
2019,Belgium,Belgium
2020,Belgium,Belgium
2021,Belgium,Belgium
2022,Brazil,Brazil


## 2. Ranking period used for each edition

In [4]:
periods = rk[["year", "semester", "ranking_date"]].drop_duplicates().sort_values("ranking_date")
ed = pd.DataFrame({"edition": config.EDITIONS})
ed["edition_label"] = ed["edition"].map(config.EDITION_LABELS)
ed["edition_date"] = pd.to_datetime(ed["edition"].map(config.EXPECTED_EDITION_DATES))
ed = pd.merge_asof(ed.sort_values("edition_date"), periods, left_on="edition_date", right_on="ranking_date",
                   direction="backward")
display(ed)
check("Every edition uses that year's semester 1 ranking",
      bool(((ed["year"] == ed["edition_date"].dt.year) & (ed["semester"] == 1)).all()))

,edition,edition_label,edition_date,year,semester,ranking_date
0,15,FIFA 15,2014-09-18,2014,1,2014-06-30
1,16,FIFA 16,2015-09-21,2015,1,2015-06-30
2,17,FIFA 17,2016-09-20,2016,1,2016-06-30
3,18,FIFA 18,2017-09-18,2017,1,2017-06-30
4,19,FIFA 19,2018-08-21,2018,1,2018-06-30
5,20,FIFA 20,2019-09-19,2019,1,2019-06-30
6,21,FIFA 21,2020-09-23,2020,1,2020-06-30
7,22,FIFA 22,2021-09-23,2021,1,2021-06-30
8,23,FIFA 23,2022-09-26,2022,1,2022-06-30
9,24,FC 24,2023-09-22,2023,1,2023-06-30


PASS - Every edition uses that year's semester 1 ranking 


True

## 3. Map FIFA nationality names to the ranking

In [5]:
panel = pd.read_parquet(config.CLEAN / "fifa_panel.parquet", columns=["nationality_name", "edition"])
coh = pd.read_parquet(config.CLEAN / "cohort.parquet", columns=["player_id", "nationality_name", "entry_edition"])
cohort_n = coh["nationality_name"].value_counts()
fifa_nations = pd.DataFrame({"fifa_nation": sorted(panel["nationality_name"].dropna().unique())})
fifa_nations["cohort_n"] = fifa_nations["fifa_nation"].map(cohort_n).fillna(0).astype(int)

ov = pd.read_csv(config.NATION_NAME_OVERRIDES, dtype=str).fillna("")
check("Override file has the expected columns",
      set(["fifa_nation", "ranking_team", "tm_country", "confederation"]) <= set(ov.columns))
check("Override file lists each FIFA nation once", not ov["fifa_nation"].duplicated().any())
ovi = ov.set_index("fifa_nation")

name_to_key = rk.drop_duplicates("team").set_index("team")["key"]
def rank_key(n):
    o = ovi["ranking_team"].get(n, "")
    if o:
        return name_to_key.get(o), "override"
    if n in name_to_key.index:
        return name_to_key[n], "exact"
    return None, "unmapped"
fifa_nations[["ranking_key", "ranking_method"]] = [rank_key(n) for n in fifa_nations["fifa_nation"]]
display(fifa_nations["ranking_method"].value_counts().to_frame("nations"))

bad_ov = ov[(ov["ranking_team"] != "") & ~ov["ranking_team"].isin(name_to_key.index)]
check("Every override ranking_team exists in the ranking", bad_ov.empty, f"{bad_ov['ranking_team'].tolist()}")

unm = fifa_nations[fifa_nations["ranking_method"] == "unmapped"]
if len(unm):
    sugg = {n: process.extractOne(n, list(name_to_key.index), scorer=fuzz.WRatio) for n in unm["fifa_nation"]}
    display(unm.assign(suggestion=unm["fifa_nation"].map(lambda n: f"{sugg[n][0]} ({sugg[n][1]:.0f})")))
check("Every FIFA nation with a cohort entrant maps to the ranking",
      unm["cohort_n"].sum() == 0, f"{int(unm['cohort_n'].sum())} entrants unmapped")
latest_name = rk.sort_values("ranking_date").drop_duplicates("key", keep="last").set_index("key")["team"]
fifa_nations["ranking_team_latest"] = fifa_nations["ranking_key"].map(latest_name)

PASS - Override file has the expected columns 
PASS - Override file lists each FIFA nation once 


,nations
ranking_method,
exact,180
override,8


PASS - Every override ranking_team exists in the ranking | []
PASS - Every FIFA nation with a cohort entrant maps to the ranking | 0 entrants unmapped


## 4. Nation rank by edition

In [6]:
nr = fifa_nations[["fifa_nation", "ranking_key"]].merge(ed[["edition", "edition_label", "edition_date", "year", "semester", "ranking_date"]], how="cross")
nr = nr.merge(rk[["key", "year", "semester", "rank", "team"]].rename(columns={"key": "ranking_key", "team": "ranking_team"}),
              on=["ranking_key", "year", "semester"], how="left")
nr = nr.rename(columns={"year": "ranking_year", "semester": "ranking_semester", "rank": "nation_rank"})
nr["nation_rank"] = nr["nation_rank"].astype("Int64")
check("nation_rank: one row per (nation, edition)", not nr.duplicated(["fifa_nation", "edition"]).any())

no_rank = nr[nr["nation_rank"].isna() & nr["ranking_key"].notna()]
print("Mapped nations without a rank in some edition (not ranked in that period):")
display(no_rank.groupby("fifa_nation")["edition_label"].agg(list))

# Coverage: cohort entrants with a nation rank at entry
ce = coh.merge(nr[["fifa_nation", "edition", "nation_rank"]], left_on=["nationality_name", "entry_edition"],
               right_on=["fifa_nation", "edition"], how="left")
cov_rank = ce["nation_rank"].notna().mean()
check("Cohort entrants with a nation rank at entry >= 99%", cov_rank >= 0.99, f"{cov_rank:.2%} of {len(ce):,}")
display(ce[ce["nation_rank"].isna()]["nationality_name"].value_counts().head(10).to_frame("entrants without rank"))

PASS - nation_rank: one row per (nation, edition) 
Mapped nations without a rank in some edition (not ranked in that period):


fifa_nation
Gibraltar    [FIFA 15, FIFA 16, FIFA 17]
Kosovo                [FIFA 15, FIFA 16]
Name: edition_label, dtype: object

PASS - Cohort entrants with a nation rank at entry >= 99% | 99.97% of 11,519


,entrants without rank
nationality_name,
Kosovo,3


## 5. ISO3 codes

In [7]:
fifa_nations[["iso3", "iso3_method"]] = [nations.iso3(n) for n in fifa_nations["fifa_nation"]]
display(fifa_nations["iso3_method"].value_counts().to_frame("nations"))
check("Every FIFA nation has an ISO3 code", fifa_nations["iso3"].notna().all(),
      str(fifa_nations.loc[fifa_nations["iso3"].isna(), "fifa_nation"].tolist()))
uk = fifa_nations[fifa_nations["fifa_nation"].isin(nations.UK_HOME_NATIONS)]
display(uk[["fifa_nation", "cohort_n", "iso3"]])
shared = fifa_nations[fifa_nations["iso3"].duplicated(keep=False)].sort_values("iso3")
print("ISO3 codes shared by more than one FIFA nation:")
display(shared[["fifa_nation", "iso3", "cohort_n"]])

,nations
iso3_method,
exact,145
manual,43


PASS - Every FIFA nation has an ISO3 code | []


,fifa_nation,cohort_n,iso3
52,England,1501,GBR
125,Northern Ireland,63,GBR
148,Scotland,283,GBR
185,Wales,97,GBR


ISO3 codes shared by more than one FIFA nation:


,fifa_nation,iso3,cohort_n
52,England,GBR,1501
125,Northern Ireland,GBR,63
148,Scotland,GBR,283
185,Wales,GBR,97


## 6. Confederation (from Transfermarkt)
`national_teams.csv` has confederations (UEFA, CAF, …). `countries.csv` only has continents; its `europa`, `afrika` and `asien`
are used as a fallback, but `amerika` mixes CONMEBOL and CONCACAF and is not used. A filled `confederation` column in the
override file wins over both.

In [8]:
nt = pd.read_csv(config.RAW_TM / "national_teams.csv")
tc = pd.read_csv(config.RAW_TM / "countries.csv")
nt_conf = nt.drop_duplicates("country_name").set_index("country_name")["confederation"]
tc_conf = tc.drop_duplicates("country_name").set_index("country_name")["confederation"].map(nations.TM_CONTINENT_TO_CONFED)

def confed(n):
    o = ovi["confederation"].get(n, "")
    if o:
        return o, (ovi["note"].get(n, "") or "override")
    tmn = ovi["tm_country"].get(n, "") or n
    if tmn in nt_conf.index:
        return nt_conf[tmn], "TM national_teams"
    if tmn in tc_conf.index and pd.notna(tc_conf[tmn]):
        return tc_conf[tmn], "TM countries"
    return None, "missing"
fifa_nations[["confederation", "confederation_source"]] = [confed(n) for n in fifa_nations["fifa_nation"]]
display(fifa_nations["confederation_source"].value_counts().to_frame("nations"))

big = fifa_nations[fifa_nations["cohort_n"] >= config.CHART_MIN_NATION_N]
miss_conf = big[big["confederation"].isna()]
print(f"Nations with cohort n >= {config.CHART_MIN_NATION_N}: {len(big)}; missing confederation: {len(miss_conf)}")
display(miss_conf[["fifa_nation", "cohort_n"]])
share_conf = big.loc[big["confederation"].notna(), "cohort_n"].sum() / big["cohort_n"].sum()
if len(miss_conf):
    NOTES.append(f"{len(miss_conf)} chart-9 nations (n ≥ {config.CHART_MIN_NATION_N}) have no confederation in the Transfermarkt tables: "
                 + ", ".join(miss_conf["fifa_nation"]) + ". Left blank; fill the `confederation` column in "
                 "nation_name_overrides.csv if the team approves a hand lookup.")
display(big.groupby("confederation", dropna=False)["fifa_nation"].agg(["size", list]))

,nations
confederation_source,
TM national_teams,118
missing,65
hand lookup: CAF member,5


Nations with cohort n >= 15: 61; missing confederation: 0


,fifa_nation,cohort_n


,size,list
confederation,,
AFC,4,"[Australia, Japan, Korea Republic, Saudi Arabia]"
CAF,11,"[Algeria, Cameroon, Congo DR, Côte d'Ivoire, G..."
CONCACAF,3,"[Canada, Mexico, United States]"
CONMEBOL,8,"[Argentina, Brazil, Chile, Colombia, Paraguay,..."
OFC,1,[New Zealand]
UEFA,34,"[Albania, Austria, Belgium, Bosnia and Herzego..."


## 7. Save

In [9]:
nation_iso = fifa_nations[["fifa_nation", "iso3", "iso3_method", "ranking_key", "ranking_team_latest", "ranking_method",
                           "confederation", "confederation_source", "cohort_n"]]
nation_rank = nr[["fifa_nation", "edition", "edition_label", "edition_date", "ranking_year", "ranking_semester",
                  "ranking_date", "ranking_key", "ranking_team", "nation_rank"]]
check("nation_iso: one row per FIFA nation", nation_iso["fifa_nation"].is_unique)
nation_iso.to_parquet(config.CLEAN / "nation_iso.parquet", index=False)
nation_rank.to_parquet(config.CLEAN / "nation_rank.parquet", index=False)
print("nation_iso.parquet:", nation_iso.shape, "| nation_rank.parquet:", nation_rank.shape)
display(nation_rank[nation_rank["edition"] == 15].sort_values("nation_rank").head(8))

PASS - nation_iso: one row per FIFA nation 
nation_iso.parquet: (188, 9) | nation_rank.parquet: (1880, 10)


,fifa_nation,edition,edition_label,edition_date,ranking_year,ranking_semester,ranking_date,ranking_key,ranking_team,nation_rank
640,Germany,15,FIFA 15,2014-09-18,2014,1,2014-06-30,GER,Germany,1
60,Argentina,15,FIFA 15,2014-09-18,2014,1,2014-06-30,ARG,Argentina,2
1180,Netherlands,15,FIFA 15,2014-09-18,2014,1,2014-06-30,NED,Netherlands,3
360,Colombia,15,FIFA 15,2014-09-18,2014,1,2014-06-30,COL,Colombia,4
150,Belgium,15,FIFA 15,2014-09-18,2014,1,2014-06-30,BEL,Belgium,5
1800,Uruguay,15,FIFA 15,2014-09-18,2014,1,2014-06-30,URU,Uruguay,6
220,Brazil,15,FIFA 15,2014-09-18,2014,1,2014-06-30,BRA,Brazil,7
1580,Spain,15,FIFA 15,2014-09-18,2014,1,2014-06-30,ESP,Spain,8


## 8. Write to `reports/data_quality.md`

In [10]:
checks = pd.DataFrame(CHECKS)
ed_md = ed.assign(edition_date=ed["edition_date"].dt.strftime("%Y-%m-%d"), ranking_date=ed["ranking_date"].dt.strftime("%Y-%m-%d"))
body = f'''
### Ranking period used per edition (semester 1 = 30 June, semester 2 = 31 December)

{dq.df_to_md(ed_md[["edition_label", "edition_date", "year", "semester", "ranking_date"]])}

### Name mapping

- FIFA nations: {len(fifa_nations)}; mapped to the ranking: {int(fifa_nations["ranking_key"].notna().sum())}
  ({", ".join(f"{k} {v}" for k, v in fifa_nations["ranking_method"].value_counts().items())}).
- Cohort entrants with a nation rank at entry: {cov_rank:.2%}.
- ISO3: {", ".join(f"{k} {v}" for k, v in fifa_nations["iso3_method"].value_counts().items())}; England, Scotland, Wales and
  Northern Ireland → GBR.
- Confederation: {", ".join(f"{k} {v}" for k, v in fifa_nations["confederation_source"].value_counts().items())}.
  Chart-9 nations (n ≥ {config.CHART_MIN_NATION_N}): {len(big)}, of which {len(miss_conf)} have no confederation
  ({share_conf:.1%} of their entrants covered).

### Notes

{chr(10).join("- " + n for n in NOTES) if NOTES else "- none"}

### Checks

{dq.df_to_md(checks)}
'''
dq.write_section("04", "04 FIFA ranking", body)
print("Written to", config.DATA_QUALITY.relative_to(ROOT))

Written to reports/data_quality.md


## 9. Summary

In [11]:
display(checks)
for n in NOTES:
    print(" -", n)
assert (checks["Result"] == "PASS").all(), checks[checks["Result"] == "FAIL"]

,Check,Result,Detail
0,Ranking covers every half-year 2014–2023,PASS,1992-12-31 to 2024-12-31
1,"No duplicate (key, period) rows in 2014–2023",PASS,4 duplicate rows elsewhere (logged)
2,Ranks are positive integers,PASS,
3,Every edition uses that year's semester 1 ranking,PASS,
4,Override file has the expected columns,PASS,
5,Override file lists each FIFA nation once,PASS,
6,Every override ranking_team exists in the ranking,PASS,[]
7,Every FIFA nation with a cohort entrant maps t...,PASS,0 entrants unmapped
8,"nation_rank: one row per (nation, edition)",PASS,
9,Cohort entrants with a nation rank at entry >=...,PASS,"99.97% of 11,519"


 - 4 duplicated (team, period) rows in the raw ranking, all outside 2014–2023: Montenegro 2007 S1, Montenegro 2007 S2. Identical copies; they are never used.
